# Oscar winners for the AWARD_Movie table
Written by: Capleton

This notebook makes a clean CSV file that fits the table AWARD_Movie in our PostgreSQL database.

## STEP 1: import os and pandas
We need pandas to prepare the data before it goes into the database, and os to save the file for the import in pgAdmin.

In [ ]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## STEP 2: read the file
The original file stays unchanged. If something goes wrong in the database, we can always build the table again from the full data.

In [ ]:
df = pd.read_csv(r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data\oscarAward.csv")

print("Rows and columns:", df.shape)
print(df.head())

## STEP 3: keep only the winners with a film
The database should only hold data that answers our research questions. An award without a film can never be linked to a row in MOVIE, so it has no place in a table with a foreign key to MOVIE.

In [ ]:
winners = df[df["winner"] == True]
winners = winners.dropna(subset=["film"])

print("Rows before:", len(df))
print("Rows after:", len(winners))

## STEP 4: keep only the movie awards
In our ERD every type of award has its own table. AWARD_Movie holds the Oscars for the film itself. The directing and acting Oscars go to AWARD_Director and AWARD_Actor, so every Oscar is stored in exactly one table and nothing is stored twice.

In [ ]:
is_director = winners["category"].str.startswith("DIRECTING")
is_actor = winners["category"].str.startswith(("ACTOR", "ACTRESS"))
award_movie = winners[~is_director & ~is_actor].copy()

print("Rows after:", len(award_movie))
print("Number of categories:", award_movie["category"].nunique())

## STEP 5: rename the columns
The column names in the CSV must be the same as in the table in PostgreSQL. Otherwise pgAdmin does not know which value goes into which column.

In [ ]:
award_movie = award_movie.rename(columns={"name": "name_nominee", "film": "film_title"})

print(list(award_movie.columns))

## STEP 6: clean the film titles and make the match key
Some old titles in the source data contain a ;. It is a leftover and not part of the real title, so we remove it to keep the data in the database clean.

film_title_clean is the title in small letters, a hyphen becomes a space, dots are removed and double spaces become one. film_title stays as it is, so the real title with capitals is still stored.

match_key is film_title_clean plus the year of the film, for example no country for old men_2007. The database needs it to fill the foreign key movie_id: the Oscar data has no movie_id, so we look up this key in MOVIE. We add the year because different films can have the same title, like a remake. The year of the film is year_ceremony minus 1, because the Oscars are given the year after the film came out. MOVIE makes the same key from its title and the year of its release date.

In [ ]:
award_movie["film_title"] = award_movie["film_title"].str.replace(";", "").str.strip()
award_movie["film_title_clean"] = award_movie["film_title"].str.lower().str.replace("-", " ", regex=False).str.replace(".", "", regex=False).str.split().str.join(" ")

award_movie["match_key"] = award_movie["film_title_clean"] + "_" + (award_movie["year_ceremony"] - 1).astype(str)

print(award_movie[["film_title", "film_title_clean", "match_key"]].head())
print("Titles with ; left:", award_movie["film_title"].str.contains(";").sum())

## STEP 7: sort the table
A fixed order gives every award the same movie_award_id every time. If the numbers change, the rows in the database no longer match the file.

In [ ]:
award_movie = award_movie.sort_values(["year_ceremony", "category", "film_title"]).reset_index(drop=True)

print(award_movie[["year_ceremony", "category", "name_nominee", "film_title", "film_title_clean"]].head())

## STEP 8: give every award a movie_award_id
movie_award_id is the primary key of AWARD_Movie. The database needs it to tell every row apart, so it must be unique and can never be empty.

In [ ]:
award_movie["movie_award_id"] = award_movie.index + 1

print("First movie_award_id:", award_movie["movie_award_id"].min())
print("Last movie_award_id:", award_movie["movie_award_id"].max())
print("All unique:", award_movie["movie_award_id"].is_unique)

## STEP 9: add an empty movie_id
movie_id is the foreign key to MOVIE in our ERD. The column must exist to fit the table, but the number is not in the Oscar data. It is filled later with the match on title and year.

In [ ]:
award_movie["movie_id"] = None

print(award_movie.head())

## STEP 10: keep only the ERD columns
The CSV must have exactly the same columns, as the table in PostgreSQL. The primary key comes first, then the foreign keys, so the most important columns are easy to find. winner is removed because every row is a winner, and year_film because it is always year_ceremony minus 1. Storing the same information twice goes against normalization. film_title_clean and match_key are not in the ERD. They stay in the file for now, because they are needed to fill movie_id. They are removed when movie_id is filled, before the import in pgAdmin.

In [ ]:
award_movie = award_movie[["movie_award_id", "movie_id", "year_ceremony",
                              "ceremony", "category", "name_nominee", "film_title", "film_title_clean", "match_key"]]

print(list(award_movie.columns))

## STEP 11: save as CSV
This file is imported into AWARD_Movie in pgAdmin. index=False prevents an extra column with row numbers, because that column does not exist in the table and the import would fail.

In [ ]:
folder = r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data"
new_file = os.path.join(folder, "award_movie_clean.csv")

award_movie.to_csv(new_file, index=False)

print("Saved:", new_file)
print("File exists:", os.path.exists(new_file))